# Kaggle Connectivity and EDA

Validation notebook for Overlay #1: Kaggle Ingestion.

This notebook checks Kaggle connectivity, inspects MinIO-backed raw/conformed/curated artifacts, performs lightweight EDA, and can regenerate the curated summary artifact.

In [1]:
from __future__ import annotations

import io
import json
import os
import subprocess
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd
from minio import Minio

In [2]:
cwd = Path.cwd().resolve()
repo_root_candidates = [cwd, cwd.parent, Path('/home/jovyan')]
repo_root = next((candidate for candidate in repo_root_candidates if (candidate / 'config').exists()), cwd)
config_path = repo_root / 'config' / 'kaggle_jobs.json'
example_config_path = repo_root / 'config' / 'kaggle_jobs.example.json'

if not config_path.exists():
    raise FileNotFoundError(
        f'Missing {config_path}. Copy {example_config_path.name} to kaggle_jobs.json and edit it first.'
    )

with config_path.open('r', encoding='utf-8') as handle:
    config = json.load(handle)

enabled_jobs = [job for job in config['jobs'] if job.get('enabled')]
if not enabled_jobs:
    raise ValueError('No enabled jobs found in kaggle_jobs.json')

job = enabled_jobs[0]
job

{'name': 'sample_kaggle_csv_ingestion',
 'enabled': True,
 'dataset': 'fedesoriano/stroke-prediction-dataset',
 'raw_target': 'kaggle/stroke_prediction',
 'conformed_target': 'kaggle/stroke_prediction/stroke_prediction.parquet',
 'curated_target': 'kaggle/stroke_prediction/stroke_prediction_summary.json'}

In [3]:
required_env = ['KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY']
env_status = {name: bool(os.getenv(name)) for name in required_env}
env_status

{'KAGGLE_API_TOKEN': True,
 'KAGGLE_USERNAME': True,
 'KAGGLE_KEY': True,
 'AWS_ACCESS_KEY_ID': False,
 'AWS_SECRET_ACCESS_KEY': False}

In [4]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
dataset_files = api.dataset_list_files(job['dataset']).files
[file.name for file in dataset_files]

['healthcare-dataset-stroke-data.csv']

In [5]:
endpoint_url = os.getenv('S3_ENDPOINT_URL', 'http://minio:9000')
parsed_endpoint = urlparse(endpoint_url)
client = Minio(
    parsed_endpoint.netloc or parsed_endpoint.path,
    access_key=os.getenv('AWS_ACCESS_KEY_ID', 'minioadmin'),
    secret_key=os.getenv('AWS_SECRET_ACCESS_KEY', 'minioadmin'),
    secure=parsed_endpoint.scheme == 'https',
)

raw_bucket = os.getenv('KAGGLE_RAW_BUCKET', 'raw')
conformed_bucket = os.getenv('KAGGLE_CONFORMED_BUCKET', 'conformed')
curated_bucket = os.getenv('KAGGLE_CURATED_BUCKET', 'curated')
raw_prefix = job['raw_target'].strip('/')
conformed_key = job['conformed_target'].lstrip('/')
curated_key = job['curated_target'].lstrip('/')

raw_objects = list(client.list_objects(raw_bucket, prefix=raw_prefix, recursive=True))
[obj.object_name for obj in raw_objects]

['kaggle/stroke_prediction/healthcare-dataset-stroke-data.csv']

In [6]:
if not raw_objects:
    raise FileNotFoundError('No raw objects found in MinIO. Run the ingestion step first.')

first_raw_object = raw_objects[0].object_name
raw_response = client.get_object(raw_bucket, first_raw_object)
try:
    raw_df = pd.read_csv(io.BytesIO(raw_response.read()))
finally:
    raw_response.close()
    raw_response.release_conn()

raw_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [7]:
{
    'raw_shape': raw_df.shape,
    'raw_columns': raw_df.columns.tolist(),
    'raw_null_counts': raw_df.isna().sum().to_dict(),
}

{'raw_shape': (5110, 12),
 'raw_columns': ['id',
  'gender',
  'age',
  'hypertension',
  'heart_disease',
  'ever_married',
  'work_type',
  'Residence_type',
  'avg_glucose_level',
  'bmi',
  'smoking_status',
  'stroke'],
 'raw_null_counts': {'id': 0,
  'gender': 0,
  'age': 0,
  'hypertension': 0,
  'heart_disease': 0,
  'ever_married': 0,
  'work_type': 0,
  'Residence_type': 0,
  'avg_glucose_level': 0,
  'bmi': 201,
  'smoking_status': 0,
  'stroke': 0}}

In [8]:
numeric_summary = raw_df.describe(include='number').transpose() if not raw_df.select_dtypes(include='number').empty else pd.DataFrame()
numeric_summary

,count,mean,std,min,25%,50%,75%,max
id,5110.0,36517.829354,21161.721625,67.00,17741.250,36932.000,54682.00,72940.00
age,5110.0,43.226614,22.612647,0.08,25.000,45.000,61.00,82.00
hypertension,5110.0,0.097456,0.296607,0.00,0.000,0.000,0.00,1.00
heart_disease,5110.0,0.054012,0.226063,0.00,0.000,0.000,0.00,1.00
avg_glucose_level,5110.0,106.147677,45.283560,55.12,77.245,91.885,114.09,271.74
bmi,4909.0,28.893237,7.854067,10.30,23.500,28.100,33.10,97.60
stroke,5110.0,0.048728,0.215320,0.00,0.000,0.000,0.00,1.00


## Univariate Analysis

Typical single-variable views for numeric and categorical columns.

In [ ]:
numeric_columns = raw_df.select_dtypes(include='number').columns.tolist()
if numeric_columns:
    plot_columns = numeric_columns[:4]
    raw_df[plot_columns].hist(figsize=(12, 8), bins=20, edgecolor='black')
    plt.suptitle('Univariate Distribution: Numeric Columns')
    plt.tight_layout()
else:
    print('No numeric columns available for histogram plots.')

In [9]:
categorical_columns = raw_df.select_dtypes(exclude='number').columns.tolist()
if categorical_columns:
    top_column = categorical_columns[0]
    raw_df[top_column].value_counts(dropna=False).head(10)
else:
    'No categorical columns available for aggregation.'

In [ ]:
if categorical_columns:
    plot_column = categorical_columns[0]
    value_counts = raw_df[plot_column].fillna('Missing').astype(str).value_counts().head(10)
    ax = value_counts.sort_values().plot(kind='barh', figsize=(10, 6), color='#4C78A8')
    ax.set_title(f'Univariate Distribution: {plot_column}')
    ax.set_xlabel('Count')
    ax.set_ylabel(plot_column)
    plt.tight_layout()
else:
    print('No categorical columns available for bar chart plots.')

In [10]:
conformed_response = client.get_object(conformed_bucket, conformed_key)
try:
    conformed_df = pd.read_parquet(io.BytesIO(conformed_response.read()))
finally:
    conformed_response.close()
    conformed_response.release_conn()

conformed_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke,source_object_key
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,kaggle/stroke_prediction/healthcare-dataset-st...
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1,kaggle/stroke_prediction/healthcare-dataset-st...
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,kaggle/stroke_prediction/healthcare-dataset-st...
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,kaggle/stroke_prediction/healthcare-dataset-st...
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,kaggle/stroke_prediction/healthcare-dataset-st...


## Bivariate Analysis

Typical relationship views across numeric and categorical variables.

In [ ]:
conformed_numeric = conformed_df.select_dtypes(include='number')
if conformed_numeric.shape[1] >= 2:
    corr = conformed_numeric.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr, cmap='Blues', vmin=-1, vmax=1)
    ax.set_title('Bivariate Analysis: Correlation Heatmap')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
else:
    print('Not enough numeric columns available for a correlation heatmap.')

In [ ]:
if conformed_numeric.shape[1] >= 2:
    x_col, y_col = conformed_numeric.columns[:2]
    ax = conformed_df.plot.scatter(x=x_col, y=y_col, figsize=(8, 6), alpha=0.5, color='#F58518')
    ax.set_title(f'Bivariate Analysis: {y_col} vs {x_col}')
    plt.tight_layout()
else:
    print('Not enough numeric columns available for a scatter plot.')

In [ ]:
conformed_categorical = conformed_df.select_dtypes(exclude='number').columns.tolist()
if conformed_categorical and not conformed_numeric.empty:
    category_col = conformed_categorical[0]
    numeric_col = conformed_numeric.columns[0]
    top_categories = conformed_df[category_col].fillna('Missing').astype(str).value_counts().head(8).index.tolist()
    plot_df = conformed_df[conformed_df[category_col].fillna('Missing').astype(str).isin(top_categories)].copy()
    plot_df[category_col] = plot_df[category_col].fillna('Missing').astype(str)
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_df.boxplot(column=numeric_col, by=category_col, ax=ax, rot=45)
    ax.set_title(f'Bivariate Analysis: {numeric_col} by {category_col}')
    ax.set_xlabel(category_col)
    ax.set_ylabel(numeric_col)
    fig.suptitle('')
    plt.tight_layout()
else:
    print('Need both numeric and categorical columns for the boxplot view.')

In [11]:
regenerate_summary = False

if regenerate_summary:
    subprocess.run(
        [
            'python3',
            str(repo_root / 'scripts' / 'conformed_to_curated.py'),
            '--config',
            str(config_path),
            '--job',
            job['name'],
        ],
        check=True,
        cwd=str(repo_root),
    )

curated_response = client.get_object(curated_bucket, curated_key)
try:
    curated_summary = json.loads(curated_response.read().decode('utf-8'))
finally:
    curated_response.close()
    curated_response.release_conn()

curated_summary

{'job_name': 'sample_kaggle_csv_ingestion',
 'dataset': 'fedesoriano/stroke-prediction-dataset',
 'raw_bucket': 'raw',
 'raw_prefix': 'kaggle/stroke_prediction',
 'conformed_bucket': 'conformed',
 'conformed_target': 'kaggle/stroke_prediction/stroke_prediction.parquet',
 'curated_bucket': 'curated',
 'curated_target': 'kaggle/stroke_prediction/stroke_prediction_summary.json',
 'row_count': 5110,
 'column_count': 13,
 'columns': ['id',
  'gender',
  'age',
  'hypertension',
  'heart_disease',
  'ever_married',
  'work_type',
  'residence_type',
  'avg_glucose_level',
  'bmi',
  'smoking_status',
  'stroke',
  'source_object_key'],
 'null_counts': {'id': 0,
  'gender': 0,
  'age': 0,
  'hypertension': 0,
  'heart_disease': 0,
  'ever_married': 0,
  'work_type': 0,
  'residence_type': 0,
  'avg_glucose_level': 0,
  'bmi': 201,
  'smoking_status': 0,
  'stroke': 0,
  'source_object_key': 0},
 'numeric_stats': {'id': {'count': 5110,
   'mean': 36517.82935420744,
   'min': 67,
   'max': 7294